In [93]:
#Bibitheken importieren

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [94]:
#Daten einlesen

daten = pd.read_csv("ai4i2020.csv")    # CSV-Datei in Variable "daten" einlesen

daten.head() #zeigt die ersten 5 Zeilen der Daten an um zu prüfen ob Daten korrekt eingelesen wurden

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


In [95]:
#zusätzliche Features erstellen

daten["temperatur_differenz"] = (daten["Process temperature [K]"] - daten["Air temperature [K]"])    # Temperaturunterschied zwischen Prozess und Umgebung

daten["verschleiss_last"] = (daten["Tool wear [min]"] * daten["Torque [Nm]"])    # Kombination aus Verschleiß und Belastung

In [96]:
#Welche Spalte soll vorhergesagt werden und welche Features sollen verwendet werden

zielspalte = "HDF"

erstes_feature = "Air temperature [K]"

zweites_feature = "temperatur_differenz"


lernrate = 0.1

anzahl_durchlaeufe = 1000

schwellenwert = 0.5

In [97]:
# Daten für die Modellierung auswählen

ausgewaehlte_daten = daten[[erstes_feature, zweites_feature, zielspalte]].copy()

In [98]:
#X- und y-Werte für die Modellierung erstellen

x_werte = ausgewaehlte_daten[[erstes_feature, zweites_feature]].values    # X-Werte = Eingabefeatures

y_werte = ausgewaehlte_daten[zielspalte].values    # y-Werte = Zielvariable

In [99]:
#Features standardisieren

mittelwerte = x_werte.mean(axis=0)    # Mittelwerte der Features berechnen (axis=0 bedeutet, dass die Mittelwerte für jede Spalte berechnet werden)

standardabweichungen = x_werte.std(axis=0)    # Standardabweichungen der Features berechnen

x_standardisiert = (x_werte - mittelwerte) / standardabweichungen    # Features standardisieren (z-Transformation)


In [100]:
#Designmatrix erstellen

einsen_spalte = np.ones((x_standardisiert.shape[0], 1))    # Spalte mit Einsen erstellen (später für theta0)

designmatrix = np.hstack([einsen_spalte,x_standardisiert])    # Designmatrix erstellen (Einsenspalte + standardisierte Features)

In [101]:
#startwerte für die Parameter (theta) festlegen

anzahl_parameter = designmatrix.shape[1]    # Anzahl der Parameter (Anzahl der Spalten in der Designmatrix) bestimmen

theta = np.zeros(anzahl_parameter)    # Parametervektor (theta) mit Nullen initialisieren (Anzahl der Parameter entspricht Anzahl der Spalten in der Designmatrix bzw. Anzahl der Features + 1 für theta0)


In [102]:
#sigmoid-Funktion

def sigmoid(wert):

    wert = np.clip(wert, -500, 500)    # Wert in einem Bereich begrenzen, um Überlauf zu vermeiden (np.exp(-wert) könnte sonst zu groß werden)

    return 1 / (1 + np.exp(-wert))    # Sigmoid-Funktion berechnen (1 / (1 + e^(-wert))) und zurückgeben

In [103]:
#restliche Funktionen

def confusion_matrix_berechnen():    # Konfusionsmatrix berechnen

    pass


def f1_score_berechnen():    # F1-Score berechnen

    pass


def trennlinie_plotten():    # Trennlinie darstellen

    pass

In [104]:
#Lossfunktion berechnen

def gewichtete_lossfunktion(designmatrix,zielwerte,parametervektor):

    linearkombination = (designmatrix @ parametervektor)    # Linearkombination aus Designmatrix und Parametervektor berechnen (z = X @ theta), für jede Zeile der Designmatrix wird die Linearkombination aus den Features und den Parametern berechnet (theta0 * 1 + theta1 * feature1 + theta2 * feature2 + ...)

    vorhergesagte_wahrscheinlichkeiten = sigmoid(linearkombination)    #Sigmoidfunktion anwenden, um die Linearkombination in Wahrscheinlichkeiten umzuwandeln (p = sigmoid(z)), große positive Werte von z führen zu p nahe 1, große negative Werte von z führen zu p nahe 0

    Grenze = 1e-15    

    vorhergesagte_wahrscheinlichkeiten = np.clip(vorhergesagte_wahrscheinlichkeiten,Grenze,1 - Grenze)    # Vorhersagen in einem Bereich begrenzen, um log(0) zu vermeiden (np.log(0) wäre mathematisch nicht definiert)

    loss = -np.mean(zielwerte * np.log(vorhergesagte_wahrscheinlichkeiten) + (1 - zielwerte) * np.log(1 - vorhergesagte_wahrscheinlichkeiten)) # -( y * log(p)+ (1-y) * log(1-p) ) binary cross-entropy loss berechnen

    return loss    # Loss zurückgeben


In [105]:
#Gradientenvektor berechnen

def gradient_berechnen(designmatrix,zielwerte,parametervektor):

    anzahl_beobachtungen = len(zielwerte)    # Anzahl der Datenpunkte bestimmen (len(...) zählt, wie viele Zeilen / Beobachtungen im Datensatz vorhanden sind)

    linearkombination = (designmatrix @ parametervektor)    # Matrix-Vektor-Produkt aus Designmatrix und Parametervektor berechnen, um die Linearkombination (z) für alle Datenpunkte zu erhalten

    vorhergesagte_wahrscheinlichkeiten = sigmoid(linearkombination)    # Sigmoidfunktion anwenden, um die Linearkombination in Wahrscheinlichkeiten umzuwandeln (p = sigmoid(z)), große positive Werte von z führen zu p nahe 1, große negative Werte von z führen zu p nahe 0

    fehler = (vorhergesagte_wahrscheinlichkeiten - zielwerte)    # Fehler berechnen (vorhergesagte Wahrscheinlichkeiten - tatsächliche Zielwerte), für jeden Datenpunkt wird die Differenz zwischen der vorhergesagten Wahrscheinlichkeit und dem tatsächlichen Zielwert berechnet

    gradienten_vektor = (1 / anzahl_beobachtungen) * (designmatrix.T @ fehler)    # Gradientenvektor berechnen (1/m * X^T @ Fehler), Designmatrix transponieren (X^T) und mit Fehler multiplizieren, dann durch Anzahl der Beobachtungen teilen, um den Durchschnittsgradienten zu erhalten

    return gradienten_vektor    # Gradientenvektor zurückgeben

In [106]:
#Modelltraining mit Gradient Descent

def modell_trainieren(designmatrix,zielwerte,parametervektor,lernrate,anzahl_durchlaeufe):

    loss_verlauf = []    # Hier speichern wir später jeden Loss-Wert, damit können wir kontrollieren, ob das Modell besser wird


    for durchlauf in range(anzahl_durchlaeufe):     # Diese Schleife wiederholt das Lernen mehrfach

        gradienten_vektor = gradient_berechnen(designmatrix,zielwerte,parametervektor) # 1. Aktuellen Gradienten berechnen (Gradientenvektor berechnen, der angibt, in welche Richtung die Parameter angepasst werden müssen, um den Loss zu minimieren)

        parametervektor = (parametervektor - lernrate * gradienten_vektor)    # 2. Parameter anpassen (Parametervektor aktualisieren, indem der Gradientenvektor mit der Lernrate multipliziert und von den aktuellen Parametern subtrahiert wird, um in Richtung des Minimums zu gehen)

        aktueller_loss = gewichtete_lossfunktion(designmatrix,zielwerte,parametervektor)    # 3. Neuen Loss berechnen, um zu sehen, ob die Anpassung der Parameter den Loss verringert hat (Lossfunktion mit den aktualisierten Parametern aufrufen)

        loss_verlauf.append(aktueller_loss)    # 4. Loss speichern (Loss-Verlauf aktualisieren, damit wir später sehen können, ob der Loss kleiner wird)

    return parametervektor, loss_verlauf    # Am Ende geben wir zurück: 1. die gelernten Theta-Werte, 2. den gesamten Loss-Verlauf

In [107]:
#Vorhersagen berechnen

def vorhersagen_berechnen(designmatrix,parametervektor,schwellenwert):

    linearkombination = (designmatrix @ parametervektor)    # Matrix-Vektor-Produkt aus Designmatrix und Parametervektor berechnen, um die Linearkombination (z) für alle Datenpunkte zu erhalten

    vorhergesagte_wahrscheinlichkeiten = sigmoid(linearkombination)    # Sigmoidfunktion anwenden, um die Linearkombination in Wahrscheinlichkeiten umzuwandeln (p = sigmoid(z)), große positive Werte von z führen zu p nahe 1, große negative Werte von z führen zu p nahe 0

    vorhergesagte_klassen = (vorhergesagte_wahrscheinlichkeiten >= schwellenwert).astype(int)    #wenn die vorhergesagte Wahrscheinlichkeit größer oder gleich dem Schwellenwert ist, wird die Vorhersage als Klasse 1 (positiv) klassifiziert, andernfalls als Klasse 0 (negativ), .astype(int) konvertiert die booleschen Werte in 0 und 1

    return vorhergesagte_wahrscheinlichkeiten, vorhergesagte_klassen    # Vorhersagen zurückgeben: 1. die vorhergesagten Wahrscheinlichkeiten, 2. die vorhergesagten Klassen basierend auf dem Schwellenwert

In [108]:
# TESTZEILE


#loss zu Beginn berechnen
start_loss = gewichtete_lossfunktion(designmatrix,y_werte,theta)

print(start_loss)


#gradientenvektor zu Beginn berechnen
start_gradient = gradient_berechnen(designmatrix,y_werte,theta)

print(start_gradient)


#modell trainieren
trainierter_parametervektor, loss_verlauf = modell_trainieren(designmatrix,y_werte,theta,lernrate,anzahl_durchlaeufe)

print("Trainierte Theta-Werte:")
print(trainierter_parametervektor)

print("Erster Loss:")
print(loss_verlauf[0])

print("Letzter Loss:")
print(loss_verlauf[-1])


#Vorhersagen berechnen
wahrscheinlichkeiten, vorhergesagte_klassen = vorhersagen_berechnen(designmatrix,trainierter_parametervektor,schwellenwert)

print("Erste 10 Wahrscheinlichkeiten:")
print(wahrscheinlichkeiten[:10])

print("Erste 10 vorhergesagte Klassen:")
print(vorhergesagte_klassen[:10])

0.6931471805599454
[ 0.4885     -0.01469549  0.02036599]
Trainierte Theta-Werte:
[-4.23240605  0.25605633 -0.66238699]
Erster Loss:
0.6695204542178433
Letzter Loss:
0.05191601405272362
Erste 10 Wahrscheinlichkeiten:
[0.00810849 0.0082121  0.00865837 0.00876894 0.0082121  0.00810849
 0.00810849 0.00810849 0.00888092 0.00853088]
Erste 10 vorhergesagte Klassen:
[0 0 0 0 0 0 0 0 0 0]
